In [1]:
#Importing libraries
import os
import pandas as pd
import numpy as np
import torch
from math import sqrt
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_forecasting import TimeSeriesDataSet, Baseline, TemporalFusionTransformer
from pytorch_forecasting.metrics import SMAPE, RMSE, MAE, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.data.encoders import NaNLabelEncoder

C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\lightning_fabric\__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_forecasting\models\base_model.py:24: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
# ---------------------------
# CONFIG
# ---------------------------
CSV_PATH = "daily_all_billers2_expanded_with_billers.csv"  # provided dataset path
TARGETS = ["total_transactions", "total_amount"]  # column names expected in CSV
DATE_COL = "txn_date"         # must exist in CSV
BILLER_COL = "biller_id"  # must exist in CSV
MAX_ENCODER_LENGTH = 30   # how many past days to feed
MAX_PREDICTION_LENGTH = 7 # forecast horizon
BATCH_SIZE = 128
NUM_EPOCHS = 20
VAL_PCT = 0.2

In [3]:
# ---------------------------
# Basic checks and load
# ---------------------------
df = pd.read_csv(CSV_PATH)
# parse dates
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

In [4]:
df.head()

,txn_date,total_transactions,total_amount,total_biller_fee,biller_id
0,2025-08-01,2212.0,1.687518e+06,2694.62,BILLER_1
1,2025-08-02,2164.0,1.703566e+06,2549.33,BILLER_1
2,2025-08-03,2143.0,1.595472e+06,2479.06,BILLER_1
3,2025-08-04,1934.0,1.336879e+06,2151.63,BILLER_1
4,2025-08-05,2096.0,1.597213e+06,2447.75,BILLER_1


In [5]:
agg = df.copy(deep=True)

In [6]:
# ---------------------------
# Build continuous time index per biller
# ---------------------------
# We'll create a global time_idx (days since min date)
# 1) Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL])

In [7]:
# 2) Sort properly
df = df.sort_values([BILLER_COL, DATE_COL])

In [8]:
# 3) Build biller-wise continuous time index (VERY IMPORTANT)
df["time_idx"] = df.groupby(BILLER_COL).cumcount().astype("int32")

In [9]:
# 4) Create calendar categorical features
df["day_of_week"] = df[DATE_COL].dt.dayofweek.astype(str)
df["is_weekend"] = df[DATE_COL].dt.dayofweek.isin([5, 6]).astype(str)
df["month"] = df[DATE_COL].dt.month.astype("float32")

In [10]:
# 5) Convert reals to float32
df["total_transactions"] = df["total_transactions"].astype("float32")
df["total_amount"] = df["total_amount"].astype("float32")

In [11]:
print(df.dtypes)

txn_date              datetime64[ns]
total_transactions           float32
total_amount                 float32
total_biller_fee             float64
biller_id                     object
time_idx                       int32
day_of_week                   object
is_weekend                    object
month                        float32
dtype: object


In [12]:
agg =df.copy(deep=True)

In [13]:
# Sort
agg = agg.sort_values([BILLER_COL, "txn_date"]).reset_index(drop=True)

In [14]:
agg["time_idx"] = agg.groupby(BILLER_COL).cumcount()


In [15]:
# ---------------------------
# Create lag & rolling features (simple)
# ---------------------------
# Create lags and rolling means per biller for both targets
def add_lags_and_rolls(df, group_col, col, lags=(1,7,14), rolls=(7,14)):
    df = df.copy()
    for lag in lags:
        df[f"{col}_lag{lag}"] = df.groupby(group_col)[col].shift(lag)
    for r in rolls:
        df[f"{col}_roll{r}"] = df.groupby(group_col)[col].shift(1).rolling(window=r, min_periods=1).mean().reset_index(level=0, drop=True)
    return df

for t in TARGETS:
    agg = add_lags_and_rolls(agg, BILLER_COL, t, lags=(1,7), rolls=(7,14))

# fill NaNs for model; lags will be NaN at start
agg.fillna(0, inplace=True)

In [16]:
# ---------------------------
# Dataset definition (shared for both targets but target column will change)
# ---------------------------
# Define static categorical list: biller ids
agg[BILLER_COL] = agg[BILLER_COL].astype(str)
agg = agg.dropna().reset_index(drop=True)


In [17]:
agg.head()

,txn_date,total_transactions,total_amount,total_biller_fee,biller_id,time_idx,day_of_week,is_weekend,month,total_transactions_lag1,total_transactions_lag7,total_transactions_roll7,total_transactions_roll14,total_amount_lag1,total_amount_lag7,total_amount_roll7,total_amount_roll14
0,2024-01-01,11144.549805,8729299.00,14185.37,BILLER_1,0,0,False,1.0,0.000000,0.0,0.000000,0.000000,0.00,0.0,0.000000e+00,0.000000e+00
1,2024-01-01,3961.510010,2970658.25,4670.48,BILLER_1,1,0,False,1.0,11144.549805,0.0,11144.549805,11144.549805,8729299.00,0.0,8.729299e+06,8.729299e+06
2,2024-01-01,2577.000000,2023808.25,3459.03,BILLER_1,2,0,False,1.0,3961.510010,0.0,7553.029907,7553.029907,2970658.25,0.0,5.849979e+06,5.849979e+06
3,2024-01-01,1720.569946,1415255.75,2132.77,BILLER_1,3,0,False,1.0,2577.000000,0.0,5894.353271,5894.353271,2023808.25,0.0,4.574588e+06,4.574588e+06
4,2024-01-01,2546.600098,2153473.00,3266.20,BILLER_1,4,0,False,1.0,1720.569946,0.0,4850.907440,4850.907440,1415255.75,0.0,3.784755e+06,3.784755e+06


In [18]:
min_len = MAX_ENCODER_LENGTH + MAX_PREDICTION_LENGTH
agg = agg.groupby(BILLER_COL).filter(lambda g: len(g) >= min_len).reset_index(drop=True)


In [19]:
# choose a split point (e.g., last 20 timesteps for validation)
max_time = agg["time_idx"].max()
split_time = max_time - 40   # leave 40 rows for val

agg_train = agg[agg["time_idx"] <= split_time].copy()
agg_val   = agg[agg["time_idx"] >  split_time].copy()



In [20]:
print(agg_train.groupby("biller_id").size())
print(agg_val.groupby("biller_id").size())


biller_id
BILLER_1      999269
BILLER_10    1000593
BILLER_2      999035
BILLER_3      997723
BILLER_4     1000408
BILLER_5     1001361
BILLER_6     1001314
BILLER_7     1000955
BILLER_8      999361
BILLER_9      999941
dtype: int64
biller_id
BILLER_5    40
dtype: int64


# split train/validation on time (to avoid leakage)
max_time_idx = agg["time_idx"].max()
val_cutoff = int(max_time_idx * (1 - VAL_PCT))

training_cutoff = val_cutoff


In [21]:
def make_tsd(target_col):

    time_varying_known_reals = ["time_idx", "month"]
    time_varying_known_categoricals = ["day_of_week", "is_weekend"]

    time_varying_unknown_reals = [target_col]   # ONLY the target itself

    static_categoricals = [BILLER_COL]

    dataset = TimeSeriesDataSet(
        agg,
        time_idx="time_idx",
        target=target_col,
        group_ids=[BILLER_COL],

        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=MAX_PREDICTION_LENGTH,

        static_categoricals=static_categoricals,

        time_varying_known_categoricals=time_varying_known_categoricals,
        time_varying_known_reals=time_varying_known_reals,

        time_varying_unknown_reals=time_varying_unknown_reals,

        target_normalizer=GroupNormalizer(
            groups=[BILLER_COL],
            transformation="softplus"
        ),

        add_relative_time_idx=False,
        add_encoder_length=False,     # ← REMOVE
        add_target_scales=False,      # ← REMOVE
        allow_missing_timesteps=True,
    )

    return dataset


In [22]:
# Build datasets for each target
tsd_count = make_tsd("total_transactions")
tsd_amount = make_tsd("total_amount")


In [23]:
tsd_count_val = TimeSeriesDataSet.from_dataset(tsd_count, agg_val, predict=True)
tsd_amount_val = TimeSeriesDataSet.from_dataset(tsd_amount, agg_val, predict=True)

C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_forecasting\data\timeseries.py:1145: UserWarning: If predicting, no randomization should be possible - setting stop_randomization=True
  warnings.warn(
C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_forecasting\data\timeseries.py:1145: UserWarning: If predicting, no randomization should be possible - setting stop_randomization=True
  warnings.warn(


In [24]:
train_dataloader_count = tsd_count.to_dataloader(train=True, batch_size=64)
val_dataloader_count = tsd_count_val.to_dataloader(train=False, batch_size=64)

train_dataloader_amount = tsd_amount.to_dataloader(train=True, batch_size=64)
val_dataloader_amount = tsd_amount_val.to_dataloader(train=False, batch_size=64)


In [25]:
def train_tft(train_loader, val_loader, dataset, target_name, max_epochs=NUM_EPOCHS):

    # logger & callbacks
    logger = TensorBoardLogger("tb_logs", name=f"tft_{target_name}")
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min",
        filename=f"tft-{target_name}" + "-{epoch:02d}-{val_loss:.4f}",
    )
    early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min")
    
    # ------------------------------------
    # FIXED: build model *from dataset*
    # ------------------------------------
    tft = TemporalFusionTransformer.from_dataset(
        dataset,
        hidden_size=32,
        lstm_layers=2,
        dropout=0.1,
        attention_head_size=4,
        learning_rate=1e-3,
        loss=QuantileLoss(),
    )
    
    # ------------------------------------
    # FIXED: new Trainer format (Lightning ≥ 2)
    # ------------------------------------
    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        gradient_clip_val=0.1,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback, early_stop_callback],
        logger=logger,
    )
    
    # train
    trainer.fit(tft, train_loader, val_loader)
    
    # ------------------------------------
    # FIXED: load best checkpoint
    # (DO NOT pass dataset)
    # ------------------------------------
    if checkpoint_callback.best_model_path:
        best_tft = TemporalFusionTransformer.load_from_checkpoint(
            checkpoint_callback.best_model_path
        )
    else:
        best_tft = tft
    
    return best_tft, trainer

In [ ]:
best_count_tft, trainer_count = train_tft(
    train_dataloader_count,
    val_dataloader_count,
    tsd_count,
    "total_transactions"
)

best_amount_tft, trainer_amount = train_tft(
    train_dataloader_amount,
    val_dataloader_amount,
    tsd_amount,
    "total_amount"
)


C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_lightning\utilities\parsing.py:262: UserWarning: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
  rank_zero_warn(
C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_lightning\utilities\parsing.py:262: UserWarning: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
  rank_zero_warn(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                           

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:224: PossibleUserWarning: The dataloader, val_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 12 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


C:\Users\localadmin\anaconda3\envs\tftenv1\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 12 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Epoch 0:   0%|          | 70/156245 [00:41<25:35:28,  1.70it/s, loss=298, v_num=4, train_loss_step=218.0]

In [ ]:
#BLOCK 5 — helper: evaluate on a dataloader (regression metrics) and plot overall comparison

def evaluate_on_dataloader(model, dataloader, verbose=True):
    """
    Evaluates a fitted TemporalFusionTransformer model on a dataloader.
    Returns flattened y_true, y_pred, and metrics (MAE, RMSE, MAPE, SMAPE).
    """

    # -----------------------------
    # 1) Predictions from the model
    # -----------------------------
    preds = model.predict(dataloader).detach().cpu().numpy()  # (n_samples, pred_len)

    # -----------------------------
    # 2) Get true values (official API)
    # -----------------------------
    true_vals = model._get_targets(dataloader).detach().cpu().numpy()  # (n_samples, pred_len)

    # -----------------------------
    # 3) Flatten both arrays
    # -----------------------------
    y_pred_flat = preds.reshape(-1)
    y_true_flat = true_vals.reshape(-1)

    # no NaNs in your dataset, so no masking needed
    # if needed, mask like: mask = ~np.isnan(y_true_flat)

    # -----------------------------
    # 4) Compute metrics
    # -----------------------------
    mae = np.mean(np.abs(y_true_flat - y_pred_flat))
    rmse = np.sqrt(np.mean((y_true_flat - y_pred_flat)**2))
    mape = np.mean(np.abs((y_true_flat - y_pred_flat) / (y_true_flat + 1e-9))) * 100
    smape = np.mean(
        2 * np.abs(y_pred_flat - y_true_flat) /
        (np.abs(y_true_flat) + np.abs(y_pred_flat) + 1e-9)
    ) * 100

    # -----------------------------
    # 5) Output
    # -----------------------------
    if verbose:
        print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.2f}%, SMAPE: {smape:.2f}%")

    return y_true_flat, y_pred_flat, {
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "smape": smape
    }


In [ ]:
def plot_overall(y_true, y_pred, title, n_points=400):
    plt.figure(figsize=(14,5))
    plt.plot(y_true[:n_points], label="Actual", linewidth=2)
    plt.plot(y_pred[:n_points], label="Predicted", linewidth=2)
    plt.title(title)
    plt.xlabel("Flattened samples")
    plt.ylabel(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
#BLOCK 6 — evaluate both models on validation dataloaders and plot
print("Evaluating txn_count model on validation set:")
y_true_count, y_pred_count, metrics_count = evaluate_on_dataloader(best_count_tft, val_dataloader_count)
plot_overall(y_true_count, y_pred_count, "Transaction Count (Actual vs Predicted)")


batch = next(iter(val_dataloader_amount))
print(batch.keys())


batch = next(iter(val_dataloader_count))
print(type(batch))
print(batch.keys())

# Build datasets for each target
tsd_count = make_tsd("total_transactions")
tsd_amount = make_tsd("total_amount")


print(tsd_count.time_varying_known_reals)
print(tsd_count.time_varying_unknown_reals)


train_count = tsd_count.filter(lambda x: x["time_idx_last"]<= training_cutoff)
val_count   = tsd_count.filter(lambda x: x["time_idx_last"]>  training_cutoff)


train_amount = tsd_amount.filter(lambda x: x["time_idx_last"]<= training_cutoff)
val_amount = tsd_amount.filter(lambda x: x["time_idx_last"]> training_cutoff)


# Create dataloaders
from torch.utils.data import DataLoader

train_dataloader_count = train_count.to_dataloader(train=True, batch_size=64, shuffle=True)
val_dataloader_count = val_count.to_dataloader(train=False, batch_size=64)


train_dataloader_amount = DataLoader(train_amount, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_dataloader_amount = DataLoader(val_amount, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


def train_tft(train_loader, val_loader, dataset, target_name, max_epochs=NUM_EPOCHS):

    # logger & callbacks
    logger = TensorBoardLogger("tb_logs", name=f"tft_{target_name}")
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min",
        filename=f"tft-{target_name}" + "-{epoch:02d}-{val_loss:.4f}",
    )
    early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min")

    # ------------------------------------
    # FIXED: build model *from dataset*
    # ------------------------------------
    tft = TemporalFusionTransformer.from_dataset(
        dataset,
        hidden_size=32,
        lstm_layers=2,
        dropout=0.1,
        attention_head_size=4,
        learning_rate=1e-3,
        loss=QuantileLoss(),
    )

    # ------------------------------------
    # FIXED: new Trainer format (Lightning ≥ 2)
    # ------------------------------------
    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        gradient_clip_val=0.1,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback, early_stop_callback],
        logger=logger,
    )

    # train
    trainer.fit(tft, train_loader, val_loader)

    # ------------------------------------
    # FIXED: load best checkpoint
    # (DO NOT pass dataset)
    # ------------------------------------
    if checkpoint_callback.best_model_path:
        best_tft = TemporalFusionTransformer.load_from_checkpoint(
            checkpoint_callback.best_model_path
        )
    else:
        best_tft = tft

    return best_tft, trainer


# ---------------------------
# Train TFT for txn_count
# ---------------------------
print("Training TFT for txn_count ...")
best_count_tft, trainer_count = train_tft(train_dataloader_count, val_dataloader_count, tsd_count, "txn_count")


# ---------------------------
# Train TFT for txn_amount
# ---------------------------
print("Training TFT for txn_amount ...")
best_amount_tft, trainer_amount = train_tft(train_dataloader_amount, val_dataloader_amount, tsd_amount, "txn_amount")


batch = next(iter(train_dataloader_amount))

for k, v in batch.items():
    print(k, type(v))


for i in range(100):
    try:
        s = tsd_amount[i]
        # print all keys and whether None
        for k,v in s.items():
            if v is None:
                print(f"Sample {i} key {k} is NONE")
                raise SystemExit
    except Exception as e:
        print("Error at sample:", i, e)
        raise


print("Static categoricals:", tsd_amount.static_categoricals)
print("Static reals:", tsd_amount.static_reals)
print("TV known reals:", tsd_amount.time_varying_known_reals)
print("TV unknown reals:", tsd_amount.time_varying_unknown_reals)
print("TV known cats:", tsd_amount.time_varying_known_categoricals)
print("TV unknown cats:", tsd_amount.time_varying_unknown_categoricals)


cols = [
 "encoder_length","total_amount_center","total_amount_scale",
 "time_idx","month",
 "total_amount","total_amount_lag1","total_amount_lag7",
 "total_amount_roll7","total_amount_roll14",
 "day_of_week","is_weekend","blr_name"
]
missing_cols = [c for c in cols if c not in df.columns]
print("Missing columns:", missing_cols)
print("NaNs per column:\n", df[cols].isna().sum())


for i, sample in enumerate(tsd_amount):
    if sample is None:
        print("❌ Found None at index:", i)
        break
else:
    print("✅ No None found in tsd_amount")


df.groupby("blr_name").size().sort_values().head(10)

len(tsd_amount)

sample = tsd_amount[0]
for k, v in sample.items():
    print(k, type(v), 
          None if v is None else (
              v if isinstance(v, (int, float)) else v.shape
          ))
